# 🤖 Module 7: Forest vs Non-Forest Classification

Welcome to **Module 7** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Pixel-Level Feature Extraction**: Transform multispectral satellite rasters into 13-dimensional biophysical feature vectors.
2. **Train Traditional ML Baselines**: Benchmark **Random Forest**, **Support Vector Machines (SVM)**, and **XGBoost**.
3. **Feature Importance Analysis**: Discover which spectral bands and indices (NDVI, NIR, SWIR, etc.) drive forest vs. non-forest separation.
4. **Spatial Mask Prediction**: Generate full 2D forest classification masks across satellite scenes.

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_07_forest_classification.ml_classifier import (
    PixelFeatureExtractor,
    ForestMLClassifierSuite
)

%matplotlib inline
print("✅ Module 7 machine learning components loaded successfully!")

## 1. Feature Extraction: 13 Spectral & Index Features per Pixel

For each pixel in the satellite image, we compute:
- **Raw Spectral Bands**: Blue, Green, Red, NIR, SWIR
- **Vegetation Indices**: NDVI, EVI, SAVI
- **Moisture & Burn Indices**: NDWI Moisture, NBR
- **Band Ratios**: NIR/Red, Red/Green, SWIR/NIR

In [ ]:
print(f"Engineered Features ({len(PixelFeatureExtractor.FEATURE_NAMES)}):", PixelFeatureExtractor.FEATURE_NAMES)

## 2. Collect Training Pixels Across Satellite Scenes

We extract balanced pixel samples (Forest = 1, Non-Forest = 0) from training scenes.

In [ ]:
def get_forest_ground_truth(bands):
    ndvi = (bands[3] - bands[2]) / (bands[3] + bands[2] + 1e-7)
    return (ndvi >= 0.50).astype(np.uint8)

train_files = sorted(glob.glob("../dataset/train/before/*.tif"))
X_train_list, y_train_list = [], []

for f_path in train_files[:6]:
    with rasterio.open(f_path) as src:
        b = src.read()
    f_label = get_forest_ground_truth(b)
    X_sub, y_sub = PixelFeatureExtractor.extract_training_sample(b, f_label, sample_size=4000)
    X_train_list.append(X_sub)
    y_train_list.append(y_sub)

X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)
print(f"Extracted {len(X_train)} training pixels across {X_train.shape[1]} features.")

## 3. Train Classifier Suite: Random Forest vs. SVM vs. XGBoost

We train all three baseline algorithms and track their training speeds.

In [ ]:
suite = ForestMLClassifierSuite(random_state=42)
suite.train_all(X_train, y_train)

## 4. Benchmark Leaderboard on Unseen Test Imagery

Evaluate models against unseen test pixels.

In [ ]:
test_files = sorted(glob.glob("../dataset/test/before/*.tif"))
with rasterio.open(test_files[0]) as src:
    test_bands = src.read()
test_label = get_forest_ground_truth(test_bands)
X_test, y_test = PixelFeatureExtractor.extract_training_sample(test_bands, test_label, sample_size=10000)

leaderboard = suite.evaluate_all(X_test, y_test)
leaderboard

## 5. Feature Importance Analysis: What Drives Forest Detection?

Let's inspect the Gini / Gain importance scores from tree models.

In [ ]:
importances = suite.get_feature_importances()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

if "Random Forest" in importances:
    rf_s = importances["Random Forest"].head(8).sort_values(ascending=True)
    axes[0].barh(rf_s.index, rf_s.values * 100, color="#27ae60", edgecolor="black")
    axes[0].set_title("Random Forest Feature Importance (MDI)")
    axes[0].set_xlabel("Importance (%)")

if "XGBoost" in importances:
    xgb_s = importances["XGBoost"].head(8).sort_values(ascending=True)
    axes[1].barh(xgb_s.index, xgb_s.values * 100, color="#e67e22", edgecolor="black")
    axes[1].set_title("XGBoost Feature Importance (Gain)")
    axes[1].set_xlabel("Importance (%)")

plt.show()

## 6. Full-Scene Forest Mask Prediction

We generate full 2D classification masks for the entire satellite scene.

In [ ]:
def make_rgb(bands):
    def s(arr):
        p2, p98 = np.percentile(arr, (2, 98))
        return np.clip((arr - p2) / (p98 - p2 + 1e-8), 0.0, 1.0)
    return np.stack([s(bands[2]), s(bands[1]), s(bands[0])], axis=-1)

pred_rf = suite.predict_scene_mask("Random Forest", test_bands)
pred_xgb = suite.predict_scene_mask("XGBoost", test_bands)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), constrained_layout=True)
axes[0].imshow(make_rgb(test_bands))
axes[0].set_title("1. Test Scene (RGB)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(test_label, cmap="Greens_r")
axes[1].set_title("2. Ground Truth Mask", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(pred_rf, cmap="Greens_r")
axes[2].set_title("3. Random Forest Mask", fontweight="bold", color="darkgreen")
axes[2].axis("off")

axes[3].imshow(pred_xgb, cmap="Greens_r")
axes[3].set_title("4. XGBoost Mask", fontweight="bold", color="darkorange")
axes[3].axis("off")

plt.show()